# Ingestion — BGE-M3 on CUDA

The aim of this notebook is to transform legal documents into vector embeddings and upload them to Pinecone.

The embedding model used is **BGE-M3**.

## Setup

1. Attach a GPU-backed kernel.
2. Upload the `Contest_Data/` directory.
3. Upload `Apikey.env`, containing `PINECONE_API_KEY`, to the notebook’s working directory.

## Pinecone index

The `legal-rag` index must have **1,024 dimensions** to match BGE-M3’s embedding output.

## Initial chunking configuration

- Chunk size: **700 words**
- Chunk overlap: **100 words**

These are initial values and can be later evaluated against alternative configurations to determine which provides the best retrieval quality.


## 1.Extract Zip 

In [1]:
import zipfile
with zipfile.ZipFile("Contest_Data.zip") as z:
    z.extractall()

## 2.Imports

In [1]:
import hashlib
import json
import os
from collections import Counter
from pathlib import Path

import torch
import sys
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer
from pinecone import Pinecone
from tqdm.auto import tqdm  

ModuleNotFoundError: No module named 'pinecone'

In [3]:
pip install -U sentence-transformers

  Using cached tokenizers-0.23.1-cp310-abi3-win_amd64.whl.metadata (10 kB)
  Using cached safetensors-0.8.0-cp310-abi3-win_amd64.whl.metadata (4.2 kB)
   ---------------------------------------- 0.0/739.6 kB ? eta -:--:--
   -------------- ------------------------- 262.1/739.6 kB ? eta -:--:--
   ---------------------------------------- 739.6/739.6 kB 1.9 MB/s  0:00:00
   ---------------------------------------- 0.0/795.8 kB ? eta -:--:--
   -------------------------- ------------- 524.3/795.8 kB 3.1 MB/s eta 0:00:01
   ---------------------------------------- 795.8/795.8 kB 3.1 MB/s  0:00:00
   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   ------- -------------------------------- 0.8/4.0 MB 3.3 MB/s eta 0:00:01
   --------------- ------------------------ 1.6/4.0 MB 4.2 MB/s eta 0:00:01
   ----------------------- ---------------- 2.4/4.0 MB 3.6 MB/s eta 0:00:01
   ---------------------------- ----------- 2.9/4.0 MB 3.2 MB/s eta 0:00:01
   ------------------------